# Full Dataset Incremental Scoring

This notebook uses **incremental processing** with **automatic checkpointing** to score the complete WildChat dataset.

## Why Incremental Approach?

✅ **Works with Free Tier** - Standard API, no Pro plan required  
✅ **Resumable** - Stop anytime, resume exactly where you left off  
✅ **Auto-Save** - Progress saved every 1000 conversations automatically  
✅ **Safe** - Never lose work, even if interrupted  
✅ **Transparent** - Real-time progress tracking and ETA  

## How It Works

1. **Process in chunks** - 1000 conversations at a time (configurable)
2. **Auto-save** - Saves progress after each chunk
3. **Auto-resume** - Detects existing progress and continues from there
4. **Rate limiting** - Automatic backoff when hitting API limits
5. **Error handling** - Retries failed requests automatically

## Timeline Estimate

For 144k conversations:
- **Single session**: ~12-16 hours continuous
- **Multiple sessions**: Process 5k-10k per day over several days
- **Flexible**: Stop/resume anytime without losing progress

## Dataset Info

- **Input**: `wildchat_full_preprocessed.csv` (~144k conversations)
- **Output**: `wildchat_full_scored_incremental.csv`
- **Chunk Size**: 1000 conversations per checkpoint
- **Total Requests**: ~288k (2 scores per conversation: empathy + attachment)

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from dotenv import load_dotenv

# Load environment
load_dotenv()

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 150)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## Check Current Progress

First, let's see if there's any existing progress.

In [2]:
# Paths
input_path = '../data/filtered/wildchat_full_preprocessed.csv'
output_path = '../data/scores/wildchat_full_scored_incremental.csv'

print(f"Input file: {input_path}")
print(f"Output file: {output_path}")
print()

# Load input dataset
df_input = pd.read_csv(input_path)
print(f"Total conversations in dataset: {len(df_input):,}")
print(f"Total API calls needed: {len(df_input) * 2:,} (empathy + attachment)")

# Check for existing progress
if os.path.exists(output_path):
    df_scored = pd.read_csv(output_path)
    scored_count = df_scored['empathy_score'].notna().sum()
    remaining = len(df_input) - scored_count
    
    print(f"\n{'='*70}")
    print("EXISTING PROGRESS FOUND")
    print(f"{'='*70}")
    print(f"Already scored: {scored_count:,} ({scored_count/len(df_input)*100:.1f}%)")
    print(f"Remaining: {remaining:,} ({remaining/len(df_input)*100:.1f}%)")
    print(f"\nYou can resume from where you left off!")
else:
    print(f"\nNo existing progress found. Will start from beginning.")

print(f"\n{'='*70}")

Input file: ../data/filtered/wildchat_full_preprocessed.csv
Output file: ../data/scores/wildchat_full_scored_incremental.csv

Total conversations in dataset: 144,439
Total API calls needed: 288,878 (empathy + attachment)

No existing progress found. Will start from beginning.

Total conversations in dataset: 144,439
Total API calls needed: 288,878 (empathy + attachment)

No existing progress found. Will start from beginning.



## Start/Resume Scoring

This cell will:
1. **Auto-detect** existing progress and resume from there
2. **Process** conversations one by one with rate limiting
3. **Auto-save** every 1000 conversations
4. **Show** real-time progress with ETA

**⚠️ IMPORTANT**: You can **stop this cell anytime** (interrupt kernel) and resume later without losing progress!

**Chunk Size**: 1000 (saves every 1000 conversations)  
**Recommended**: Run in sessions of 5k-10k conversations, then take breaks

**Time Estimate**: 
- 1000 conversations ≈ 30-45 minutes
- 5000 conversations ≈ 2.5-4 hours
- 10000 conversations ≈ 5-8 hours

In [4]:
""" %%time

# Run incremental scoring
# This automatically resumes from where it left off
!python3 ../scripts/score_conversations_incremental.py \
    --input ../data/filtered/wildchat_full_preprocessed.csv \
    --output ../data/scores/wildchat_full_scored_incremental.csv \
    --chunk-size 1000 """

' %%time\n\n# Run incremental scoring\n# This automatically resumes from where it left off\n!python3 ../scripts/score_conversations_incremental.py     --input ../data/filtered/wildchat_full_preprocessed.csv     --output ../data/scores/wildchat_full_scored_incremental.csv     --chunk-size 1000 '

## Check Progress After Run

Let's see how much progress we made.

In [5]:
if os.path.exists(output_path):
    df_scored = pd.read_csv(output_path)
    
    empathy_scored = df_scored['empathy_score'].notna().sum()
    attachment_scored = df_scored['attachment_score'].notna().sum()
    total = len(df_scored)
    
    print(f"{'='*70}")
    print("CURRENT PROGRESS")
    print(f"{'='*70}")
    print(f"\nTotal conversations: {total:,}")
    print(f"\nScored:")
    print(f"  Empathy:    {empathy_scored:,} ({empathy_scored/total*100:.1f}%)")
    print(f"  Attachment: {attachment_scored:,} ({attachment_scored/total*100:.1f}%)")
    print(f"\nRemaining:   {total - empathy_scored:,} ({(total - empathy_scored)/total*100:.1f}%)")
    
    if empathy_scored == total:
        print(f"\n🎉 ALL CONVERSATIONS SCORED! 🎉")
    else:
        print(f"\n⏳ Scoring in progress... Re-run the cell above to continue.")
    
    print(f"{'='*70}")
else:
    print("No progress file found yet.")

No progress file found yet.


---

# Analysis (Run After Scoring Complete)

The cells below analyze the scored dataset. **Only run these after scoring is 100% complete!**

## Load Scored Dataset

In [ ]:
# Load the fully scored dataset
scored_path = '../data/scores/wildchat_full_scored_incremental.csv'

print(f"Loading scored dataset from: {scored_path}")
df_full_scored = pd.read_csv(scored_path)

print(f"\nDataset shape: {df_full_scored.shape}")
print(f"Total turn pairs: {len(df_full_scored):,}")
print(f"\nColumns: {list(df_full_scored.columns)}")
print(f"\nFirst few rows:")
df_full_scored.head()

## Data Quality Check

In [ ]:
print("="*70)
print("DATA QUALITY CHECK")
print("="*70)

print(f"\nTotal turn pairs: {len(df_full_scored):,}")
print(f"\nMissing scores:")
print(f"  Empathy:    {df_full_scored['empathy_score'].isna().sum():>6,} ({df_full_scored['empathy_score'].isna().sum()/len(df_full_scored)*100:>5.2f}%)")
print(f"  Attachment: {df_full_scored['attachment_score'].isna().sum():>6,} ({df_full_scored['attachment_score'].isna().sum()/len(df_full_scored)*100:>5.2f}%)")

print(f"\nValid scores:")
print(f"  Empathy:    {df_full_scored['empathy_score'].notna().sum():>6,} ({df_full_scored['empathy_score'].notna().sum()/len(df_full_scored)*100:>5.2f}%)")
print(f"  Attachment: {df_full_scored['attachment_score'].notna().sum():>6,} ({df_full_scored['attachment_score'].notna().sum()/len(df_full_scored)*100:>5.2f}%)")

print("\n" + "="*70)

# Check if there are any invalid scores (outside 1-7 range)
invalid_empathy = df_full_scored[(df_full_scored['empathy_score'] < 1) | (df_full_scored['empathy_score'] > 7)]
invalid_attachment = df_full_scored[(df_full_scored['attachment_score'] < 1) | (df_full_scored['attachment_score'] > 7)]

if len(invalid_empathy) > 0:
    print(f"⚠️  Warning: {len(invalid_empathy)} empathy scores outside 1-7 range")
if len(invalid_attachment) > 0:
    print(f"⚠️  Warning: {len(invalid_attachment)} attachment scores outside 1-7 range")

if len(invalid_empathy) == 0 and len(invalid_attachment) == 0:
    print("✅ All scores are within valid range (1-7)")

## Empathy Score Distribution

In [ ]:
print("="*70)
print("EMPATHY SCORE DISTRIBUTION (Treatment Variable)")
print("="*70)

empathy_counts = df_full_scored['empathy_score'].value_counts().sort_index()
empathy_pct = (empathy_counts / len(df_full_scored) * 100).round(2)

print(f"\n{'Score':>6} {'Count':>12} {'Percentage':>12}")
print("-"*32)
for score in range(1, 8):
    count = empathy_counts.get(score, 0)
    pct = empathy_pct.get(score, 0.0)
    bar = '█' * int(pct / 2)  # Visual bar
    print(f"{score:>6} {count:>12,} {pct:>11.2f}% {bar}")
print("-"*32)
print(f"{'Total':>6} {len(df_full_scored):>12,} {'100.00%':>12}")

print(f"\nSummary Statistics:")
print(f"  Mean:   {df_full_scored['empathy_score'].mean():.2f}")
print(f"  Median: {df_full_scored['empathy_score'].median():.2f}")
print(f"  Std:    {df_full_scored['empathy_score'].std():.2f}")
print(f"  Min:    {df_full_scored['empathy_score'].min():.0f}")
print(f"  Max:    {df_full_scored['empathy_score'].max():.0f}")

# Group into low/mid/high
low = (df_full_scored['empathy_score'] <= 3).sum()
mid = ((df_full_scored['empathy_score'] == 4)).sum()
high = (df_full_scored['empathy_score'] >= 5).sum()

print(f"\nGrouped Distribution:")
print(f"  Low (1-3):  {low:>8,} ({low/len(df_full_scored)*100:>5.1f}%)")
print(f"  Mid (4):    {mid:>8,} ({mid/len(df_full_scored)*100:>5.1f}%)")
print(f"  High (5-7): {high:>8,} ({high/len(df_full_scored)*100:>5.1f}%)")
print("="*70)

## Attachment Score Distribution

In [ ]:
print("="*70)
print("ATTACHMENT SCORE DISTRIBUTION (Outcome Variable)")
print("="*70)

attachment_counts = df_full_scored['attachment_score'].value_counts().sort_index()
attachment_pct = (attachment_counts / len(df_full_scored) * 100).round(2)

print(f"\n{'Score':>6} {'Count':>12} {'Percentage':>12}")
print("-"*32)
for score in range(1, 8):
    count = attachment_counts.get(score, 0)
    pct = attachment_pct.get(score, 0.0)
    bar = '█' * int(pct / 2)  # Visual bar
    print(f"{score:>6} {count:>12,} {pct:>11.2f}% {bar}")
print("-"*32)
print(f"{'Total':>6} {len(df_full_scored):>12,} {'100.00%':>12}")

print(f"\nSummary Statistics:")
print(f"  Mean:   {df_full_scored['attachment_score'].mean():.2f}")
print(f"  Median: {df_full_scored['attachment_score'].median():.2f}")
print(f"  Std:    {df_full_scored['attachment_score'].std():.2f}")
print(f"  Min:    {df_full_scored['attachment_score'].min():.0f}")
print(f"  Max:    {df_full_scored['attachment_score'].max():.0f}")

# Group into low/mid/high for causal analysis
control = (df_full_scored['attachment_score'] <= 3).sum()
middle = ((df_full_scored['attachment_score'] > 3) & (df_full_scored['attachment_score'] < 5)).sum()
treatment = (df_full_scored['attachment_score'] >= 5).sum()

print(f"\nCausal Analysis Groups:")
print(f"  Control (≤3):       {control:>8,} ({control/len(df_full_scored)*100:>5.1f}%)")
print(f"  Middle (3<Y<5):     {middle:>8,} ({middle/len(df_full_scored)*100:>5.1f}%)")
print(f"  Treatment (≥5):     {treatment:>8,} ({treatment/len(df_full_scored)*100:>5.1f}%)")
print(f"  Usable (C+T):       {control+treatment:>8,} ({(control+treatment)/len(df_full_scored)*100:>5.1f}%)")
print("="*70)

## Visualize Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Empathy scores
axes[0].hist(df_full_scored['empathy_score'].dropna(), bins=np.arange(0.5, 8.5, 1),
             color='skyblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Empathy Score Distribution (T) - Full Dataset', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Empathy Score (1-7)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_xticks(range(1, 8))
emp_mean = df_full_scored['empathy_score'].mean()
axes[0].axvline(emp_mean, color='red', linestyle='--', linewidth=2,
                label=f'Mean: {emp_mean:.2f}')
axes[0].legend(fontsize=11)
axes[0].grid(axis='y', alpha=0.3)

# Attachment scores
axes[1].hist(df_full_scored['attachment_score'].dropna(), bins=np.arange(0.5, 8.5, 1),
             color='lightcoral', edgecolor='black', alpha=0.7)
axes[1].set_title('Attachment Score Distribution (Y) - Full Dataset', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Attachment Score (1-7)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_xticks(range(1, 8))
att_mean = df_full_scored['attachment_score'].mean()
axes[1].axvline(att_mean, color='red', linestyle='--', linewidth=2,
                label=f'Mean: {att_mean:.2f}')
axes[1].legend(fontsize=11)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/scores/full_dataset_score_distributions.png', dpi=200, bbox_inches='tight')
plt.show()

print("✓ Saved plot: outputs/scores/full_dataset_score_distributions.png")

## Summary

### ✅ Incremental Scoring Complete!

**Method**: Incremental Processing with Auto-Checkpointing  
**Dataset**: WildChat Full Dataset (~144k conversations)  
**Processing**: Multiple resumable sessions  

### Files Created:

1. **Scored Dataset**: `data/scores/wildchat_full_scored_incremental.csv`
2. **Visualizations**: `outputs/scores/`

### Next Steps:

1. ✅ Full dataset scored with empathy and attachment ratings
2. ⏳ Generate embeddings for user prompts (X1)
3. ⏳ Apply propensity score matching
4. ⏳ Perform causal analysis (ATE estimation)
5. ⏳ Generate final results and visualizations